In [ ]:
# Yolov8 without hand labeling.

In [ ]:
pip install ultralytics

In [ ]:
from ultralytics import YOLO
import pandas as pd
import os

df = pd.read_csv("movies_10y_metadata.csv")
model = YOLO("yolov8s.pt")

image_paths = df["image_path"].tolist()
persons = [None] * len(df)

batch_size = 64

for start in range(0, len(image_paths), batch_size):
    batch_paths = image_paths[start:start + batch_size]

    valid_batch = []
    valid_indices = []

    for j, path in enumerate(batch_paths):
        if os.path.exists(path):
            valid_batch.append(path)
            valid_indices.append(start + j)

    if not valid_batch:
        continue

    results = model(valid_batch, verbose=False)

    for idx, result in zip(valid_indices, results):
        classes = result.boxes.cls.tolist() if result.boxes is not None else []
        count = sum(1 for cls in classes if int(cls) == 0)
        persons[idx] = count

    print(f"Processed {min(start + batch_size, len(image_paths))}/{len(image_paths)}")

df["person_count"] = persons
df.to_csv("movies_10y_metadata_with_count1.csv", index=False)